# Preset-Katalog für die Website

Excel bleibt die Quelle. Der Snapshot ist eine Momentaufnahme pro Toolstand; Namen dürfen sich ändern. Keine Szenario-API, keine Migration der HVAC-Verbraucher.

Die aktive Datei wird in `scripts/output_catalog.py` mit `PRESET_SNAPSHOT` ausdrücklich ausgewählt. Neue Version: aus dem passenden Schema-Export erzeugen, Unterschiede und redaktionelle Zuordnungen prüfen, erst dann Auswahl ändern. Vor Freigabe DEV kennzeichnen; alte Releases nicht überschreiben.

In [1]:
from pathlib import Path
import sys, json, tempfile
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/output_catalog.py').exists())
sys.path.insert(0, str(ROOT))
from scripts.output_catalog import export_preset_snapshot, build_presets, PRESET_SNAPSHOT
source = ROOT / 'data/exports/schemas/schema_1.13.1_dev.xlsx'


## Reproduzierbaren Export prüfen
Die Prüfung schreibt nur in einen temporären Ordner. Für einen bewusst gewählten neuen Stand dieselbe Exportfunktion mit dem neuen Zielpfad aufrufen.

In [2]:
with tempfile.TemporaryDirectory() as temp:
    check = export_preset_snapshot(source, Path(temp) / 'presets.json')
    assert check.read_bytes() == PRESET_SNAPSHOT.read_bytes()
data = json.loads(PRESET_SNAPSHOT.read_text(encoding='utf-8'))
print(data['tool_version'], len(data['tables']), sum(len(t['presets']) for t in data['tables']))
# Reale Belegungen: Null und False dürfen nicht als leere Zellen verschwinden.
pv = next(t for t in data['tables'] if t['name'] == 'PV_system')
none = next(p for p in pv['presets'] if p['name'] == 'Keine PV')
assert none['values']['PV_roof_share'] == 0
assert none['values']['PV_is_used'] is False
assert any(v is None for t in data['tables'] for p in t['presets'] for v in p['values'].values())

import openpyxl
wb = openpyxl.load_workbook(source, read_only=False, data_only=True)
try:
    for table in data['tables']:
        sheet = wb[table['name']]
        for preset in table['presets']:
            for key, address in preset['cells'].items():
                assert sheet[address].value == preset['values'][key], (table['name'], address)
finally:
    wb.close()
print('Alle exportierten Werte gegen die Quellzellen geprüft.')


1.13.1_dev 26 285


Alle exportierten Werte gegen die Quellzellen geprüft.


## Website erzeugen
`build_presets` liest ausschließlich den ausgewählten Snapshot. Die Notizen werden für Katalog und Fachseiten gemeinsam genutzt. Neue oder entfernte Tabellennamen lösen eine Prüfung aus; keine dauerhaften Preset-IDs. Namensänderungen erfordern einen redaktionellen Link-/Kommentarabgleich.

Die Zuordnung `pages` kann auch ein Abschnittsziel wie `hvac-heizung` bezeichnen. HVAC bindet vier getrennte Hilfen jeweils am Ende von Heizung, Kühlung, Warmwasser und Lüftung ein; die Texte bleiben zentral gepflegt.

In [3]:
print(build_presets())
import subprocess

C:\Users\schneids\code\peexcel\website\assets\preset-catalog.qmd


In [4]:
result = subprocess.run(['quarto', 'render', 'preset-catalog.qmd'], cwd=ROOT / 'website', capture_output=True, text=True)
assert result.returncode == 0, result.stderr
print('preset-catalog: gerendert')

preset-catalog: gerendert


In [5]:
result = subprocess.run(['quarto', 'render', 'presets.qmd'], cwd=ROOT / 'website', capture_output=True, text=True)
assert result.returncode == 0, result.stderr
print('presets: gerendert')

presets: gerendert


In [6]:
result = subprocess.run(['quarto', 'render', 'glossary.qmd'], cwd=ROOT / 'website', capture_output=True, text=True)
assert result.returncode == 0, result.stderr
print('glossary: gerendert')

glossary: gerendert


In [7]:
result = subprocess.run(['quarto', 'render', 'hvac.qmd'], cwd=ROOT / 'website', capture_output=True, text=True)
assert result.returncode == 0, result.stderr
print('hvac: gerendert')

hvac: gerendert


In [8]:
result = subprocess.run(['quarto', 'render', 'pv.qmd'], cwd=ROOT / 'website', capture_output=True, text=True)
assert result.returncode == 0, result.stderr
print('pv: gerendert')

pv: gerendert


In [9]:
result = subprocess.run(['quarto', 'render', 'usage.qmd'], cwd=ROOT / 'website', capture_output=True, text=True)
assert result.returncode == 0, result.stderr
print('usage: gerendert')

usage: gerendert


In [10]:
result = subprocess.run(['quarto', 'render', 'model.qmd'], cwd=ROOT / 'website', capture_output=True, text=True)
assert result.returncode == 0, result.stderr
print('model: gerendert')

model: gerendert


In [11]:
result = subprocess.run(['quarto', 'render', 'envelope.qmd'], cwd=ROOT / 'website', capture_output=True, text=True)
assert result.returncode == 0, result.stderr
print('envelope: gerendert')

envelope: gerendert


In [12]:
result = subprocess.run(['quarto', 'render', 'systems.qmd'], cwd=ROOT / 'website', capture_output=True, text=True)
assert result.returncode == 0, result.stderr
print('systems: gerendert')

systems: gerendert


In [13]:
result = subprocess.run(['quarto', 'render', 'mobility.qmd'], cwd=ROOT / 'website', capture_output=True, text=True)
assert result.returncode == 0, result.stderr
print('mobility: gerendert')

mobility: gerendert


In [14]:
result = subprocess.run(['quarto', 'render', 'climate.qmd'], cwd=ROOT / 'website', capture_output=True, text=True)
assert result.returncode == 0, result.stderr
print('climate: gerendert')

climate: gerendert


In [15]:
result = subprocess.run(['quarto', 'render', 'declaration.qmd'], cwd=ROOT / 'website', capture_output=True, text=True)
assert result.returncode == 0, result.stderr
print('declaration: gerendert')

declaration: gerendert


## Abnahme und Pflege

Vorschau: `/preset-catalog.html`. Suche, Gruppenfilter, Detailwerte und Links aus Fachseiten prüfen. Platzhalter bleiben sichtbar; fehlende Werte werden nicht mit Null verwechselt. Der Export enthält Quellzellen und Herkunfts-Hash. Keine Änderung der Quelldatei und kein automatischer COM-Aufruf.

Der JSON-Bestand ersetzt keine Excel-Datei; der Website-Build exportiert nicht neu. Nach Wertänderungen bewusst neu exportieren und fachlich abnehmen. Zahlenwerte nicht in den redaktionellen Notizen pflegen. Interne Diskussion in der privaten Hypothesis-Gruppe, Veröffentlichung nach Abnahme.